# Uncertainty-Aware 3D Object Detection — Full Pipeline
# Sırayla tüm hücreleri çalıştır: Setup → Training → Evaluation

In [ ]:
# ============================================================
# 1. GPU CHECK + DRIVE MOUNT
# ============================================================
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/uncertainty_3d_detection'
!mkdir -p {PROJECT_DIR}/checkpoints
!mkdir -p {PROJECT_DIR}/data
print('Drive mounted.')

In [ ]:
# ============================================================
# 2. INSTALL DEPENDENCIES + EXTRACT CODE
# ============================================================
import subprocess, sys, os

!pip install pyyaml easydict tensorboardX scikit-learn tqdm -q

# Her zaman zip'ten yeniden cikart (guncel kodu garantile)
PROJECT_DIR = '/content/drive/MyDrive/uncertainty_3d_detection'
ZIP_PATH = f'{PROJECT_DIR}/uncertainty_3d_detection.zip'
ZIP_PATH_ALT = '/content/drive/MyDrive/uncertainty_3d_detection.zip'
zip_to_use = ZIP_PATH if os.path.exists(ZIP_PATH) else ZIP_PATH_ALT

!rm -rf /content/uncertainty_3d_detection

if os.path.exists(zip_to_use):
    print(f'Zip bulundu: {zip_to_use}')
    !unzip -o -q {zip_to_use} -d /content/
    # Ic ice klasor varsa duzelt
    inner = '/content/uncertainty_3d_detection/uncertainty_3d_detection'
    if os.path.exists(inner) and os.path.exists(os.path.join(inner, 'tools')):
        !mv {inner}/* /content/uncertainty_3d_detection/
        !rm -rf {inner}
    print('Proje kodu kuruldu!')
else:
    raise FileNotFoundError(f'ZIP BULUNAMADI! Yukle: {ZIP_PATH}')

sys.path.insert(0, '/content/uncertainty_3d_detection')
print('Setup tamam.')

In [ ]:
# ============================================================
# 3. KITTI DATASET + CONFIG
# ============================================================
import yaml, os, sys
import numpy as np

PROJECT_DIR = '/content/drive/MyDrive/uncertainty_3d_detection'

# KITTI verisini bul
possible_kitti = [
    '/content/drive/MyDrive/uncertainty_3d_detection/data/kitti',
    '/content/drive/MyDrive/data/kitti',
]
KITTI_DIR = None
for p in possible_kitti:
    if os.path.exists(os.path.join(p, 'training', 'velodyne')):
        KITTI_DIR = p
        break

if KITTI_DIR is None:
    raise FileNotFoundError('KITTI bulunamadi! Drive\'a yukle: .../uncertainty_3d_detection/data/kitti/')

vel_count = len([f for f in os.listdir(os.path.join(KITTI_DIR, 'training', 'velodyne')) if f.endswith('.bin')])
print(f'KITTI bulundu: {KITTI_DIR} ({vel_count} velodyne)')

# ImageSets olustur
imageset_dir = os.path.join(KITTI_DIR, 'ImageSets')
os.makedirs(imageset_dir, exist_ok=True)
if not os.path.exists(os.path.join(imageset_dir, 'train.txt')):
    all_ids = list(range(7481))
    np.random.seed(0)
    np.random.shuffle(all_ids)
    train_ids = sorted(all_ids[:3712])
    val_ids = sorted(all_ids[3712:])
    with open(f'{imageset_dir}/train.txt', 'w') as f:
        f.write('\n'.join(f'{i:06d}' for i in train_ids))
    with open(f'{imageset_dir}/val.txt', 'w') as f:
        f.write('\n'.join(f'{i:06d}' for i in val_ids))
    print(f'ImageSets olusturuldu: {len(train_ids)} train, {len(val_ids)} val')
else:
    print('ImageSets zaten mevcut')

# Config
with open('/content/uncertainty_3d_detection/configs/centerpoint_kitti.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['data']['data_path'] = KITTI_DIR
config['train']['batch_size'] = 4
config['train']['num_workers'] = 2
config['paths']['checkpoint_dir'] = f'{PROJECT_DIR}/checkpoints'
config['paths']['output_dir'] = f'{PROJECT_DIR}/output'
config['paths']['log_dir'] = f'{PROJECT_DIR}/logs'
config['paths']['eval_dir'] = f'{PROJECT_DIR}/eval'
config['paths']['viz_dir'] = f'{PROJECT_DIR}/viz'

with open('/content/colab_config.yaml', 'w') as f:
    yaml.dump(config, f)

print(f"Config hazir! Epochs: {config['train']['epochs']}, BS: {config['train']['batch_size']}")

In [ ]:
# ============================================================
# 4. TRAINING — otomatik mod secimi + config versiyon kontrolu
#
# Mantik:
#  - CONFIG_VERSION degistiyse → eski checkpointler silinir (temiz baslangic)
#  - Ayni versiyonda epoch checkpoint varsa → session resume
#  - Ayni versiyonda sadece best_model.pth varsa → finetune
#  - Hicbir sey yoksa → sifirdan baslat
# ============================================================
import os, glob, shutil, re

PROJECT_DIR = '/content/drive/MyDrive/uncertainty_3d_detection'
ckpt_dir = f'{PROJECT_DIR}/checkpoints'
eval_dir = f'{PROJECT_DIR}/eval'
viz_dir = f'{PROJECT_DIR}/viz'

# Bu versiyonu config materyal degistiginde bump et
CONFIG_VERSION = 'v3_option_a_lr0.0005_bs4'
marker_file = os.path.join(ckpt_dir, '.config_version')

# --- Versiyon kontrolu ---
os.makedirs(ckpt_dir, exist_ok=True)
needs_wipe = False
existing_files = [f for f in os.listdir(ckpt_dir) if not f.startswith('.')]

if existing_files:
    if os.path.exists(marker_file):
        with open(marker_file, 'r') as f:
            old_version = f.read().strip()
        if old_version != CONFIG_VERSION:
            print(f'Config degisti: {old_version} -> {CONFIG_VERSION}')
            needs_wipe = True
        else:
            print(f'Config ayni versiyonda: {CONFIG_VERSION}')
    else:
        print('Config marker yok — eski checkpointler temizleniyor')
        needs_wipe = True

if needs_wipe:
    print('Eski checkpointler siliniyor...')
    shutil.rmtree(ckpt_dir)
    os.makedirs(ckpt_dir, exist_ok=True)
    # Eski eval/viz sonuclarini da temizle (eski modele aitti)
    for d in [eval_dir, viz_dir]:
        if os.path.exists(d):
            shutil.rmtree(d)
            os.makedirs(d, exist_ok=True)
    print('Temizlik tamam.')

# Yeni versiyon marker'ini yaz
with open(marker_file, 'w') as f:
    f.write(CONFIG_VERSION)

# --- Mod secimi ---
# CRITICAL: epoch checkpointlerini SAYISAL sirala (string sort bug'ini onle)
# 'checkpoint_epoch_9.pth' > 'checkpoint_epoch_12.pth' as strings — bug!
def _epoch_num(p):
    m = re.search(r'checkpoint_epoch_(\d+)\.pth', os.path.basename(p))
    return int(m.group(1)) if m else -1

epoch_ckpts = sorted(
    glob.glob(os.path.join(ckpt_dir, 'checkpoint_epoch_*.pth')),
    key=_epoch_num,
)
best_model = os.path.join(ckpt_dir, 'best_model.pth')

if epoch_ckpts:
    last_ckpt = epoch_ckpts[-1]
    print(f'\nEpoch checkpoint bulundu: {os.path.basename(last_ckpt)} (epoch {_epoch_num(last_ckpt)})')
    print(f'Mevcut epoch checkpointleri: {[_epoch_num(c) for c in epoch_ckpts]}')
    print('Mod: SESSION RESUME — kaldigi yerden devam')
    !cd /content/uncertainty_3d_detection && python tools/train.py --config /content/colab_config.yaml 2>&1
elif os.path.exists(best_model):
    print('\nbest_model.pth bulundu')
    print('Mod: FINETUNE — taze scheduler ile')
    for f in ['latest.pth', 'final_model.pth']:
        fp = os.path.join(ckpt_dir, f)
        if os.path.exists(fp):
            os.remove(fp)
    !cd /content/uncertainty_3d_detection && python tools/train.py --config /content/colab_config.yaml --finetune {best_model} 2>&1
else:
    print('\nMod: FRESH START — sifirdan baslatiliyor')
    !cd /content/uncertainty_3d_detection && python tools/train.py --config /content/colab_config.yaml 2>&1

In [ ]:
# ============================================================
# 5. EVALUATION — Evidential
# ============================================================
PROJECT_DIR = '/content/drive/MyDrive/uncertainty_3d_detection'
!cd /content/uncertainty_3d_detection && python tools/evaluate.py \
    --config /content/colab_config.yaml \
    --checkpoint {PROJECT_DIR}/checkpoints/best_model.pth \
    --mode evidential \
    --visualize \
    --num_viz 20 2>&1

In [ ]:
# ============================================================
# 6. EVALUATION — MC Dropout
# ============================================================
PROJECT_DIR = '/content/drive/MyDrive/uncertainty_3d_detection'
!cd /content/uncertainty_3d_detection && python tools/evaluate.py \
    --config /content/colab_config.yaml \
    --checkpoint {PROJECT_DIR}/checkpoints/best_model.pth \
    --mode mc_dropout \
    --mc_passes 20 2>&1

In [ ]:
# ============================================================
# 7. VIEW RESULTS
# ============================================================
from IPython.display import Image, display
import glob, os

PROJECT_DIR = '/content/drive/MyDrive/uncertainty_3d_detection'

# Evaluation plots
print('=== EVALUATION PLOTS ===')
for img_path in sorted(glob.glob(f'{PROJECT_DIR}/eval/*.png')):
    print(f'\n--- {os.path.basename(img_path)} ---')
    display(Image(filename=img_path, width=600))

# BEV visualizations
print('\n=== BEV VISUALIZATIONS ===')
for img_path in sorted(glob.glob(f'{PROJECT_DIR}/viz/*.png'))[:10]:
    print(f'\n--- {os.path.basename(img_path)} ---')
    display(Image(filename=img_path, width=800))

In [ ]:
# ============================================================
# 8. DISCONNECT (Colab GPU saatini bosa harcama)
# ============================================================
from google.colab import runtime
runtime.unassign()